In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History
import pandas as pd
import numpy as np

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [3]:
query = sa.text("""
    SELECT *
    FROM meta_vars
    WHERE network_id = 11;
""")

with engine.connect() as conn:
    var_db = pd.read_sql(query, conn)
    
var_db

,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user
0,534,11,%,None,relative_humidity,time: maximum,None,RHx,Relative Humidity (Max.),relative_humidity_maximum,2025-02-11 16:03:39.747374,crmp
1,506,11,m s-1,None,wind_speed,time: mean,None,WindSpeedms,Wind Speed (Mean),wind_speed_mean,2025-02-11 16:03:39.747374,crmp
2,538,11,m s-1,None,wind_speed,time: minimum,None,Wind_n,Wind Speed (Min.),wind_speed_minimum,2025-02-11 16:03:39.747374,crmp
3,504,11,%,None,relative_humidity,time: mean,None,RH,Relative Humidity (Mean),relative_humidity_mean,2025-02-11 16:03:39.747374,crmp
4,505,11,celsius,None,dew_point_temperature,time: mean,None,DewPtC,Dew Point Temperature (Mean),dew_point_temperature_mean,2025-02-11 16:03:39.747374,crmp
5,502,11,millibar,None,air_pressure,time: point,None,Pressurembar,Air Pressure (Point),air_pressure_point,2025-02-11 16:03:39.747374,crmp
6,501,11,mm,None,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,Rainmm,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,2025-02-11 16:03:39.747374,crmp
7,529,11,celsius,None,air_temperature,time: mean,None,Tm,Temperature (Mean),air_temperature_mean,2025-02-11 16:03:39.747374,crmp
8,530,11,celsius,None,air_temperature,time: maximum,None,Tx,Temperature (Max.),air_temperature_maximum,2025-02-11 16:03:39.747374,crmp
9,531,11,celsius,None,air_temperature,time: minimum,None,Tn,Temperature (Min.),air_temperature_minimum,2025-02-11 16:03:39.747374,crmp


### Add 5 variables

In [4]:
from pycds import Variable
from sqlalchemy.inspection import inspect

####  Variable ORM does NOT map 1-to-1 to the table columns. It uses different attribute names.
# help(Variable)
Variable.__table__.columns.keys()

inspect(Variable).mapper.column_attrs.keys()

['id',
 'name',
 'unit',
 'standard_name',
 'cell_method',
 'precision',
 'description',
 'display_name',
 'short_name',
 'network_id',
 'mod_time',
 'mod_user']

In [9]:
variables_data = [
    dict(
        network_id=11,
        unit='celsius',
        standard_name='surface_temperature',
        cell_method='time: mean (interval: 1 hour)',
        description='Surface temperature',
        name='Surface_Temp',
        display_name='Surface temperature',
        short_name='surface_temperature',
    ),
    # dict(
    #     network_id=11,
    #     unit='celsius',
    #     standard_name='soil_temperature',
    #     cell_method='time: mean (interval: 1 hour)',
    #     description='Soil temperature - 25cm',
    #     name='Soil_Temp_25cm',
    #     display_name='Soil temperature - 25cm',
    #     short_name='soil_temperature_25cm',
    # ),
    # dict(
    #     network_id=11,
    #     unit='celsius',
    #     standard_name='soil_temperature',
    #     cell_method='time: mean (interval: 1 hour)',
    #     description='Soil temperature - 50cm',
    #     name='Soil_Temp_50cm',
    #     display_name='Soil temperature - 50cm',
    #     short_name='soil_temperature_50cm',
    # ),
    # dict(
    #     network_id=11,
    #     unit='celsius',
    #     standard_name='soil_temperature',
    #     cell_method='time: mean (interval: 1 hour)',
    #     description='Soil temperature - 75cm',
    #     name='Soil_Temp_75cm',
    #     display_name='Soil temperature - 75cm',
    #     short_name='soil_temperature_75cm',
    # ),
    # dict(
    #     network_id=11,
    #     unit='mm',
    #     standard_name='lwe_thickness_of_precipitation_amount_raw',
    #     cell_method='time: sum',
    #     description='Raw depth of water-equivalent rain',
    #     name='Rainmm_raw',
    #     display_name='Rainmm_raw',
    #     short_name='lwe_thickness_of_precipitation_amount_sum_raw',
    # ),
    # dict(
    #     network_id=11,
    #     unit='m3/m3',
    #     standard_name='liquid_water_content_of_soil_layer',
    #     cell_method='time: mean (interval: 1 hour)',
    #     description='Calibrated liquid water content of soil - 15 cm',
    #     name='Water_Content_Cal_m3_m3_15_cm',
    #     display_name='Calibrated Soil Liquid Water Content - 15 cm',
    #     short_name='soil_liquid_water_content_cal_15cm',
    # ),
    # dict(
    #     network_id=11,
    #     unit='celsius',
    #     standard_name='air_temperature',
    #     cell_method='time: mean',
    #     description='air_temperature_mean - 30cm',
    #     name='TempC_30_cm',
    #     display_name='Temperature - 30cm (Mean)',
    #     short_name='air_temperature_mean_30cm',
    # ),
    # dict(
    #     network_id=11,
    #     unit='%',
    #     standard_name='relative_humidity',
    #     cell_method='time: mean',
    #     description='Relative Humidity (Mean) - 30cm',
    #     name='RH_30cm',
    #     display_name='Relative Humidity (Mean) - 30cm',
    #     short_name='relative_humidity_mean_30cm',
    # ),
    # dict(
    #     network_id=11,
    #     unit='celsius',
    #     standard_name='dew_point_temperature',
    #     cell_method='time: mean',
    #     description='Dew Point Temperature (Mean) - 30cm',
    #     name='DewPt_30cm',
    #     display_name='Dew Point Temperature (Mean) - 30cm',
    #     short_name='dew_point_temperature_mean_30cm',
    # )
]

In [10]:
# Query existing variables in ONE database call

existing = (
    session.query(Variable)
    .filter(
        Variable.network_id == 11,
        Variable.name.in_([v["name"] for v in variables_data]),
    )
    .all()
)

existing_names = {v.name for v in existing}

# Create only the missing ones
to_insert = [
    Variable(**v)
    for v in variables_data
    if v["name"] not in existing_names
]

session.add_all(to_insert)
session.commit()